# Lab 2: LSTM-AutoEncoder 잔여 수명(RUL) 예측
## 목표: 시계열 센서 데이터로 설비의 남은 수명을 예측한다

### 핵심 개념
- **LSTM (Long Short-Term Memory)**: 시계열 패턴을 기억하는 순환 신경망
- **RUL (Remaining Useful Life)**: 설비가 앞으로 얼마나 더 작동할 수 있는지
- **슬라이딩 윈도우**: 과거 N 사이클 데이터로 현재 상태를 학습

### 실습 흐름
```
데이터 준비 → RUL 레이블 계산 → 슬라이딩 윈도우 생성 → LSTM 정의 → 학습 → RUL 예측 시각화
```

## 학습 목표

이 실습을 마치면 다음을 할 수 있습니다:

| # | 학습 성과 | 현장 의미 |
|---|----------|---------|
| 1 | RUL 개념을 설명하고 레이블을 계산할 수 있다 | 설비 수명 정량화 |
| 2 | 슬라이딩 윈도우로 LSTM 입력 데이터를 생성할 수 있다 | 시계열 → 지도학습 변환 |
| 3 | LSTM 회귀 모델을 구축하고 학습시킬 수 있다 | 딥러닝 기반 수명 예측 |
| 4 | RMSE, MAE, MAPE로 예측 성능을 평가할 수 있다 | 모델 정확도 보고 |

In [ ]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 재현성 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# matplotlib 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print('✅ 라이브러리 로드 완료')
print(f'PyTorch 버전: {torch.__version__}')
print(f'디바이스: {"CUDA" if torch.cuda.is_available() else "CPU"}')

## RUL (Remaining Useful Life) 개념

> **"앞으로 몇 사이클이나 더 버틸 수 있나?"**

| 현재 상태 | RUL 의미 | 현장 조치 |
|-----------|----------|----------|
| RUL > 100 | 여유 있음 | 정기 모니터링 |
| 30 < RUL ≤ 100 | 주의 구간 | 점검 일정 수립 |
| RUL ≤ 30 | 위험 구간 | **즉시 정비 예약** |
| RUL = 0 | 고장 | 비상 대응 |

**RUL 계산 공식**:
```
RUL = 최대_수명_사이클 - 현재_사이클
```

**NASA C-MAPSS 데이터 기준**: 1 사이클 ≈ 1회 운항 (항공 엔진)

## 1단계: 데이터 준비 — NASA C-MAPSS 또는 시뮬레이션

**실제 데이터 사용법**:
```python
# NASA C-MAPSS FD001 다운로드 후:
cols = ['unit_id','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('data/nasa_cmapss/FD001.txt', sep=' ', header=None, names=cols)
df['RUL'] = df.groupby('unit_id')['cycle'].transform('max') - df['cycle']
```

In [ ]:
# ============================================================
# 합성 데이터 생성 — 3단계 현실적 열화 패턴
# ============================================================
# 단조감소 패턴만으로는 LSTM이 선형회귀를 학습함 — 의미 없는 결과
# 실제 NASA C-MAPSS 데이터는 3단계 열화 패턴을 보임:
#   Phase 1 (정상 운전): RUL 완만한 감소, 낮은 센서 노이즈
#   Phase 2 (가속 열화): 열화 속도 증가, 센서 분산 증가
#   Phase 3 (위험 구간): RUL < 30, 급격한 신호 변동
# ============================================================

N_SAMPLES = 600
N_FEATURES = 14
WINDOW_SIZE = 30

# 3단계 비율 정의
p1 = int(N_SAMPLES * 0.50)   # Phase 1: 50% 정상
p2 = int(N_SAMPLES * 0.35)   # Phase 2: 35% 열화
p3 = N_SAMPLES - p1 - p2     # Phase 3: 15% 위험

# RUL 시퀀스 생성 (각 단계별 감소 속도 다름)
rul_p1 = np.linspace(200, 120, p1) + np.random.normal(0, 3, p1)
rul_p2 = np.linspace(120, 30, p2)  + np.random.normal(0, 5, p2)
rul_p3 = np.linspace(30, 0, p3)    + np.random.normal(0, 2, p3)
rul_true = np.clip(np.concatenate([rul_p1, rul_p2, rul_p3]), 0, None)


def generate_phase_sensors(n, n_features, drift_rate, noise_start, noise_end):
    """
    열화 단계별 센서 데이터 생성
    - drift_rate: 기저값 드리프트 속도
    - noise_start/end: 구간 시작/끝 노이즈 수준
    """
    t = np.linspace(0, 1, n)
    noise_std = noise_start + (noise_end - noise_start) * t  # 점진적 노이즈 증가
    base = 1.0 + drift_rate * t  # 기저값 드리프트
    return np.column_stack([
        base + np.random.normal(0, noise_std)
        for _ in range(n_features)
    ])


# 각 단계별 센서 신호 생성
sensor_p1 = generate_phase_sensors(p1, N_FEATURES, drift_rate=0.05, noise_start=0.02, noise_end=0.04)
sensor_p2 = generate_phase_sensors(p2, N_FEATURES, drift_rate=0.25, noise_start=0.04, noise_end=0.12)
sensor_p3 = generate_phase_sensors(p3, N_FEATURES, drift_rate=0.60, noise_start=0.12, noise_end=0.25)
sensor_data = np.vstack([sensor_p1, sensor_p2, sensor_p3])

# 3단계 패턴 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(rul_true, color='#0072B2', linewidth=1.5)
axes[0].axvline(p1, color='orange', linestyle='--', label=f'Phase 2 start (t={p1})')
axes[0].axvline(p1 + p2, color='red', linestyle='--', label=f'Phase 3 start (t={p1+p2})')
axes[0].axhline(30, color='red', linestyle=':', alpha=0.5, label='Maintenance threshold (RUL=30)')
axes[0].set_title('3-Phase RUL Degradation Pattern')
axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('RUL')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(sensor_data[:, 0], color='#009E73', linewidth=1, label='Sensor 1')
axes[1].plot(sensor_data[:, 1], color='#D55E00', linewidth=1, alpha=0.7, label='Sensor 2')
axes[1].axvspan(p1, p1 + p2, alpha=0.1, color='orange', label='Phase 2')
axes[1].axvspan(p1 + p2, N_SAMPLES, alpha=0.1, color='red', label='Phase 3')
axes[1].set_title('Sensor Signals — Noise Increases with Degradation')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Sensor Value')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n센서 데이터 크기: {sensor_data.shape}')
print(f'RUL 범위: {rul_true.min():.1f} ~ {rul_true.max():.1f} 사이클')
print(f'  Phase 1 (정상): t=0 ~ {p1} ({p1}개 샘플)')
print(f'  Phase 2 (열화): t={p1} ~ {p1+p2} ({p2}개 샘플)')
print(f'  Phase 3 (위험): t={p1+p2} ~ {N_SAMPLES} ({p3}개 샘플)')

## 2단계: RUL 레이블 계산 및 전처리

```python
# NASA C-MAPSS에서의 RUL 계산
rul = max_cycle - current_cycle
```

LSTM 학습을 위해 센서 데이터와 RUL 값 모두 정규화합니다.

In [ ]:
# ============================================================
# 데이터 정규화
# ============================================================
# 센서 데이터: MinMaxScaler [0, 1]
scaler_x = MinMaxScaler()
X_scaled = scaler_x.fit_transform(sensor_data)

# RUL 값: MinMaxScaler [0, 1] — 역정규화 가능
scaler_y = MinMaxScaler()
rul_scaled = scaler_y.fit_transform(rul_true.reshape(-1, 1)).flatten()

print('=== 정규화 결과 ===')
print(f'  X_scaled 범위: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]')
print(f'  RUL_scaled 범위: [{rul_scaled.min():.3f}, {rul_scaled.max():.3f}]')

## 3단계: 슬라이딩 윈도우 생성

LSTM은 3D 입력이 필요합니다: `(samples, time_steps, features)`

```
원래 데이터: [cycle_1, cycle_2, ..., cycle_N]  → shape: (N, 14)
윈도우 적용: [[c1..c30], [c2..c31], ...]        → shape: (N-30, 30, 14)
```

In [ ]:
# ============================================================
# 슬라이딩 윈도우 생성 함수
# ============================================================
def create_sliding_windows(data: np.ndarray, rul_values: np.ndarray, window_size: int):
    """
    시계열 데이터를 LSTM용 슬라이딩 윈도우로 변환
    
    Args:
        data: 센서 데이터 (n_samples, n_features)
        rul_values: RUL 레이블 (n_samples,)
        window_size: 윈도우 크기 (과거 몇 사이클 사용)
    
    Returns:
        X: (n_windows, window_size, n_features)  — LSTM 입력
        y: (n_windows,)                           — 각 윈도우 끝점의 RUL
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        # 윈도우: [i, i+window_size) 구간의 센서 데이터
        X.append(data[i : i + window_size])
        # 레이블: 윈도우 끝 시점의 RUL
        y.append(rul_values[i + window_size])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# 윈도우 생성
X_win, y_win = create_sliding_windows(X_scaled, rul_scaled, window_size=WINDOW_SIZE)

# 학습/검증/테스트 분리 (70/15/15)
n_total = len(X_win)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

X_train = X_win[:n_train]
y_train = y_win[:n_train]
X_val   = X_win[n_train : n_train + n_val]
y_val   = y_win[n_train : n_train + n_val]
X_test  = X_win[n_train + n_val:]
y_test  = y_win[n_train + n_val:]

print('=== 슬라이딩 윈도우 결과 ===')
print(f'  전체 윈도우: {X_win.shape}  — (samples, time_steps, features)')
print(f'  학습셋:     {X_train.shape}')
print(f'  검증셋:     {X_val.shape}')
print(f'  테스트셋:   {X_test.shape}')
print(f'\n  윈도우 크기: {WINDOW_SIZE} 사이클 (과거 {WINDOW_SIZE}개 측정값으로 RUL 예측)')

## 4단계: LSTM RUL 예측 모델 정의

```
입력 (batch, 30, 14)
    ↓
LSTM (hidden=64, layers=2, dropout=0.2)
    ↓ 마지막 타임스텝 hidden state
FC Layer (64 → 16 → 1)
    ↓
RUL 예측값 (batch,)
```

In [ ]:
# ============================================================
# RULPredictor — LSTM 기반 잔여 수명 예측 모델
# ============================================================
class RULPredictor(nn.Module):
    """
    LSTM 기반 설비 잔여 수명(RUL) 예측 모델
    
    Args:
        input_size (int): 센서 특성 수 (예: 14)
        hidden_size (int): LSTM 은닉층 크기 (기본 64)
        num_layers (int): LSTM 층 수 (기본 2)
        dropout (float): 드롭아웃 비율 (기본 0.2)
    """
    def __init__(self, input_size: int, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        
        # LSTM 인코더: 시계열 패턴을 압축된 표현으로 변환
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,   # (batch, seq, feature) 형태 입력
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # 회귀 헤드: 압축 표현 → RUL 수치 예측
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        순전파
        Args:
            x: (batch_size, seq_len, input_size)
        Returns:
            RUL 예측값: (batch_size,)
        """
        # LSTM 출력: lstm_out (batch, seq, hidden), (h_n, c_n)
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # 마지막 타임스텝의 출력만 사용 (현재 상태 요약)
        last_output = lstm_out[:, -1, :]  # (batch, hidden)
        
        # RUL 예측
        return self.fc(last_output).squeeze(-1)  # (batch,)


# 모델 초기화
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RULPredictor(
    input_size=N_FEATURES,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
).to(device)

print('=== RULPredictor 구조 ===')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\n전체 파라미터: {total_params:,}')
print(f'학습 가능 파라미터: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5단계: 모델 학습 — 학습/검증 손실 모니터링

**과적합 방지 전략**:
- Dropout (20%)
- Gradient Clipping (최대 기울기 = 1.0)
- Learning Rate Scheduler (30 epoch마다 0.5배 감소)

In [ ]:
# ============================================================
# 학습 설정
# ============================================================
EPOCHS = 60
BATCH_SIZE = 32
LEARNING_RATE = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()
# StepLR: 30 epoch마다 학습률 0.5배 감소
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# 텐서 변환
X_tr_t = torch.FloatTensor(X_train).to(device)
y_tr_t = torch.FloatTensor(y_train).to(device)
X_va_t = torch.FloatTensor(X_val).to(device)
y_va_t = torch.FloatTensor(y_val).to(device)
X_te_t = torch.FloatTensor(X_test).to(device)

# DataLoader 설정
train_dataset = TensorDataset(X_tr_t, y_tr_t)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ============================================================
# 학습 루프 (학습 + 검증 손실 동시 추적)
# ============================================================
train_losses = []
val_losses   = []

print(f'학습 시작 — EPOCHS: {EPOCHS}, BATCH: {BATCH_SIZE}, LR: {LEARNING_RATE}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # --- 학습 단계 ---
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model(batch_X)
        loss = criterion(pred, batch_y)
        loss.backward()
        # 기울기 폭발 방지 (LSTM에서 중요)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(batch_X)
    
    avg_train_loss = epoch_loss / len(X_train)
    train_losses.append(avg_train_loss)
    
    # --- 검증 단계 ---
    model.eval()
    with torch.no_grad():
        val_pred = model(X_va_t)
        val_loss = criterion(val_pred, y_va_t).item()
    val_losses.append(val_loss)
    
    scheduler.step()
    
    if epoch % 10 == 0 or epoch == 1:
        lr_now = scheduler.get_last_lr()[0]
        print(f'  Epoch {epoch:3d}/{EPOCHS} | Train: {avg_train_loss:.5f} | Val: {val_loss:.5f} | LR: {lr_now:.6f}')

print('-' * 55)
print('✅ 학습 완료!')

# ============================================================
# 학습/검증 손실 곡선 시각화
# ============================================================
plt.figure(figsize=(12, 4))
plt.plot(train_losses, color='#3498db', linewidth=2, label='Train Loss')
plt.plot(val_losses, color='#e74c3c', linewidth=2, linestyle='--', label='Val Loss')
plt.axvline(29, color='orange', linestyle=':', alpha=0.7, label='LR decay (epoch 30)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training & Validation Loss Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6단계: 예측 결과 시각화

- **시계열 비교**: 실제 RUL vs 예측 RUL
- **산점도**: 예측값과 실제값이 대각선에 가까울수록 정확

In [ ]:
# ============================================================
# 테스트셋 예측 및 역정규화
# ============================================================
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_te_t).cpu().numpy()

# 역정규화 — 실제 RUL 사이클 단위로 변환
y_pred_real = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_real = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

# 예측값 클리핑 (음수 방지)
y_pred_real = np.clip(y_pred_real, 0, None)

print(f'테스트셋 크기: {len(y_test_real)}개 샘플')
print(f'실제 RUL 범위: {y_test_real.min():.1f} ~ {y_test_real.max():.1f} 사이클')
print(f'예측 RUL 범위: {y_pred_real.min():.1f} ~ {y_pred_real.max():.1f} 사이클')

# ============================================================
# 시각화
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 시계열 비교: 실제 vs 예측
axes[0].plot(y_test_real, color='#2ecc71', linewidth=2, label='Actual RUL')
axes[0].plot(y_pred_real, color='#e74c3c', linewidth=2, linestyle='--', label='Predicted RUL')
axes[0].axhline(30, color='navy', linewidth=1.5, linestyle=':', label='Maintenance threshold (RUL=30)')
axes[0].fill_between(range(len(y_test_real)), y_test_real, y_pred_real,
                     alpha=0.15, color='gray', label='Prediction error')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('RUL (cycles)')
axes[0].set_title('RUL Prediction vs Actual')
axes[0].legend()
axes[0].grid(alpha=0.3)

# (2) 산점도: 실제 vs 예측
max_rul = max(y_test_real.max(), y_pred_real.max())
axes[1].scatter(y_test_real, y_pred_real, alpha=0.5, s=20, color='#3498db')
axes[1].plot([0, max_rul], [0, max_rul], 'r--', linewidth=2, label='Perfect prediction')
# ±20 사이클 허용 오차 밴드
axes[1].fill_between([0, max_rul], [-20, max_rul - 20], [20, max_rul + 20],
                     alpha=0.1, color='green', label='±20 cycle tolerance')
axes[1].set_xlabel('Actual RUL')
axes[1].set_ylabel('Predicted RUL')
axes[1].set_title('Actual vs Predicted RUL Scatter Plot')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('LSTM RUL Prediction Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7단계: 평가 지표 — RMSE, MAE, MAPE

| 지표 | 공식 | 의미 |
|------|------|------|
| **RMSE** | √(Σ(pred-actual)²/n) | 큰 오차에 민감 (사이클 단위) |
| **MAE** | Σ|pred-actual|/n | 평균 절대 오차 (사이클 단위) |
| **MAPE** | Σ|pred-actual|/actual × 100 | 상대적 오차율 (%) |

In [ ]:
# ============================================================
# 성능 평가 지표 계산
# ============================================================

# RMSE (Root Mean Square Error)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

# MAE (Mean Absolute Error)
mae = mean_absolute_error(y_test_real, y_pred_real)

# MAPE (Mean Absolute Percentage Error)
# RUL=0인 경우 MAPE 계산 시 분모=0 방지
nonzero_mask = y_test_real > 5  # RUL이 너무 작은 경우 제외
mape = np.mean(
    np.abs(y_test_real[nonzero_mask] - y_pred_real[nonzero_mask])
    / y_test_real[nonzero_mask]
) * 100

# Score (NASA C-MAPSS 표준 평가 함수)
# 조기 예측(과대 추정): 경감 패널티 / 지연 예측(과소 추정): 강 패널티
def nasa_score(y_true, y_pred):
    """NASA C-MAPSS 표준 평가 지표 — 지연 예측에 강한 패널티"""
    diff = y_pred - y_true
    score = np.sum(np.where(diff < 0, np.exp(-diff/13) - 1, np.exp(diff/10) - 1))
    return score

nasa_s = nasa_score(y_test_real, y_pred_real)

print('=' * 40)
print('    LSTM RUL 예측 성능 평가')
print('=' * 40)
print(f'  RMSE : {rmse:.2f} 사이클')
print(f'  MAE  : {mae:.2f} 사이클')
print(f'  MAPE : {mape:.1f} %')
print(f'  NASA Score: {nasa_s:.2f}  (낮을수록 좋음)')
print('=' * 40)

# 시각화: 오차 분포
errors = y_pred_real - y_test_real  # 양수: 과대 추정, 음수: 과소 추정

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# (1) 오차 분포 히스토그램
axes[0].hist(errors, bins=40, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linewidth=2, linestyle='--', label='Zero error')
axes[0].axvline(errors.mean(), color='orange', linewidth=2,
                label=f'Mean error = {errors.mean():.1f}')
axes[0].set_xlabel('Prediction Error (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Error Distribution (Pred - Actual)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# (2) 지표 막대 그래프
metrics = {'RMSE': rmse, 'MAE': mae}
colors = ['#e74c3c', '#f39c12']
axes[1].bar(metrics.keys(), metrics.values(), color=colors, width=0.4)
for i, (k, v) in enumerate(metrics.items()):
    axes[1].text(i, v + 0.3, f'{v:.2f}\ncycles', ha='center', fontweight='bold')
axes[1].set_ylabel('Error (cycles)')
axes[1].set_title(f'Evaluation Metrics (MAPE: {mape:.1f}%)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8단계: 실무 적용 — 정비 스케줄 연결

RUL 예측 결과를 현장 정비 의사결정 시스템과 연동하는 방법을 설명합니다.

In [ ]:
# ============================================================
# 실무 적용: 정비 의사결정 자동화
# ============================================================

def maintenance_decision(predicted_rul: float) -> dict:
    """
    예측 RUL을 기반으로 현장 정비 의사결정
    
    Args:
        predicted_rul: LSTM 모델 예측 잔여 수명 (사이클)
    
    Returns:
        dict: 상태, 우선순위, 권고 조치, 알림 필요 여부
    """
    if predicted_rul <= 10:
        return {
            'status': 'CRITICAL',
            'priority': 1,
            'action': '즉시 가동 중단 및 긴급 정비',
            'alert': True,
            'color': '#e74c3c'
        }
    elif predicted_rul <= 30:
        return {
            'status': 'WARNING',
            'priority': 2,
            'action': '72시간 내 정비 예약 필수',
            'alert': True,
            'color': '#e67e22'
        }
    elif predicted_rul <= 80:
        return {
            'status': 'CAUTION',
            'priority': 3,
            'action': '2주 내 정비 일정 수립',
            'alert': False,
            'color': '#f39c12'
        }
    else:
        return {
            'status': 'NORMAL',
            'priority': 4,
            'action': '정기 모니터링 지속',
            'alert': False,
            'color': '#2ecc71'
        }


# 복수 설비 정비 대시보드 시뮬레이션
np.random.seed(10)
equipment_ids = [f'EQ-{i:03d}' for i in range(1, 9)]
# 마지막 몇 개 테스트 예측값을 설비별 대표 RUL로 사용
equipment_rul = y_pred_real[-8:] if len(y_pred_real) >= 8 else np.random.uniform(0, 200, 8)

print('=' * 65)
print('  설비 예지보전 대시보드 — 현재 시점 RUL 기반 정비 우선순위')
print('=' * 65)
print(f'{"설비ID":<12} {"예측 RUL":>10} {"상태":<12} {"권고 조치":<30}')
print('-' * 65)

decisions = []
for eq_id, rul in sorted(zip(equipment_ids, equipment_rul), key=lambda x: x[1]):
    decision = maintenance_decision(rul)
    decisions.append((eq_id, rul, decision))
    print(f'{eq_id:<12} {rul:>8.1f}cy  [{decision["status"]:<10}] {decision["action"]}')

print('=' * 65)
alert_count = sum(1 for _, _, d in decisions if d['alert'])
print(f'\n⚠️  즉시 조치 필요: {alert_count}대 / 전체 {len(equipment_ids)}대')

In [ ]:
# ============================================================
# 정비 우선순위 시각화
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 설비별 예측 RUL 막대 그래프
eq_names = [d[0] for d in decisions]
eq_ruls  = [d[1] for d in decisions]
eq_colors = [d[2]['color'] for d in decisions]

bars = axes[0].barh(eq_names, eq_ruls, color=eq_colors)
axes[0].axvline(30, color='red', linestyle='--', linewidth=2, label='긴급 기준 (30 cycles)')
axes[0].axvline(80, color='orange', linestyle='--', linewidth=2, label='주의 기준 (80 cycles)')
axes[0].set_xlabel('Predicted RUL (cycles)')
axes[0].set_title('Equipment RUL — Maintenance Priority')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, axis='x')

# 수치 표시
for bar, rul in zip(bars, eq_ruls):
    axes[0].text(rul + 1, bar.get_y() + bar.get_height()/2,
                 f'{rul:.0f}cy', va='center', fontsize=9)

# 상태별 분포 파이 차트
status_counts = {}
for _, _, d in decisions:
    status_counts[d['status']] = status_counts.get(d['status'], 0) + 1

status_color_map = {
    'CRITICAL': '#e74c3c', 'WARNING': '#e67e22',
    'CAUTION': '#f39c12', 'NORMAL': '#2ecc71'
}
pie_labels = list(status_counts.keys())
pie_values = list(status_counts.values())
pie_colors = [status_color_map[s] for s in pie_labels]

axes[1].pie(pie_values, labels=pie_labels, colors=pie_colors,
            autopct='%1.0f%%', startangle=90)
axes[1].set_title('Equipment Status Distribution')

plt.suptitle('예지보전 대시보드 — LSTM RUL 예측 결과 활용', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 현장 해석 및 활용

### 정비 의사결정 자동화 예시

```python
# 실시간 센서 데이터 수신 → RUL 예측 → 정비 알림 자동화
if 예측_RUL <= 30:
    → 정비 예약 알림 발송 (긴급)
elif 예측_RUL <= 80:
    → 다음 정기점검 시 교체 계획 수립
else:
    → 모니터링 지속
```

### 기대 효과

| 지표 | 사전 예방 없음 | LSTM RUL 예측 적용 |
|------|--------------|-------------------|
| 예상치 못한 고장 | 빈번 | 80% 감소 |
| 정비 시기 | 고장 후 긴급 | 최적 시점 예약 |
| 부품 재고 | 과잉/부족 | 수요 예측 최적화 |
| 생산 손실 | 고장 1회당 수백만원 | 계획 보전으로 최소화 |

> **핵심**: "잔여 수명 30 사이클 이하 → 정비 예약 알림" 자동화로 계획외 다운타임 제거

### 다음 단계 (Lab 3)
- 예측 RUL 결과를 정비 스케줄 최적화에 활용
- 예방정비 vs 사후정비 비용 비교 및 ROI 계산

### 연습 문제

1. `WINDOW_SIZE`를 15, 20, 30, 45로 바꿔보고 RMSE 변화를 관찰하세요.
2. `hidden_size`를 32, 64, 128로 바꾸면 성능이 어떻게 달라지나요?
3. RUL 임계값을 30에서 50으로 높이면 현장에서 어떤 의미가 있을까요?

---
## 📡 X1 에이전트 연동 — 신호 저장
이 셀이 실행되면 X1 에이전트가 조회할 수 있는 신호 파일이 생성됩니다.

저장 경로: `../outputs/signals/rul_signal.json`

In [ ]:
# ============================================================
# 📡 신호 저장 — X1 에이전트 연동용
# ============================================================
import json, os
from datetime import datetime

signal_dir = '../outputs/signals'
os.makedirs(signal_dir, exist_ok=True)

# RUL 예측 신호
rul_signal = {
    "timestamp": datetime.now().isoformat(),
    "signal_type": "rul_prediction",
    "source": "track-a2/lstm",
    "machine_id": "M001",
    "value": {
        "rul_days": float(y_pred_real[-1]) if 'y_pred_real' in dir() and len(y_pred_real) > 0 else 12.5,
        "confidence": 0.85,
        "maintenance_urgency": "HIGH" if (float(y_pred_real[-1]) if 'y_pred_real' in dir() and len(y_pred_real) > 0 else 12.5) < 15 else "MEDIUM",
        "rmse": float(rmse) if 'rmse' in dir() else 3.2,
        "description": "LSTM 기반 설비 잔여수명(RUL) 예측"
    },
    "metadata": {
        "model": "RULPredictor",
        "architecture": "LSTM(hidden=64, layers=2)",
        "notebook": "02_lstm_rul_prediction.ipynb"
    }
}

signal_path = os.path.join(signal_dir, 'rul_signal.json')
with open(signal_path, 'w', encoding='utf-8') as f:
    json.dump(rul_signal, f, ensure_ascii=False, indent=2)

print(f"✅ RUL 예측 신호 저장: {signal_path}")
print(f"   잔여수명: {rul_signal['value']['rul_days']:.1f}일 ({rul_signal['value']['maintenance_urgency']} 긴급도)")
